In [1]:
import pandas as pd
print("Setup Successful! Pandas Version:", pd__version__)

NameError: name 'pd__version__' is not defined

In [2]:
import pandas as pd
print("Setup Successful! Pandas Version:", pd.__version__)

Setup Successful! Pandas Version: 3.1.0


In [1]:
import numpy as np
import pandas as pd

# Set a random seed so you get the exact same data every time
np.random.seed(42)

# 1. Create 10,000 Users with Realistic Upward Growth
n_users = 10000
user_ids = [f"USER_{i:05d}" for i in range(1, n_users + 1)]

# Generate realistic daily signups that grow over the 6-month period
all_days = pd.date_range(start="2026-01-01", end="2026-06-30", freq="D")
growth_weights = np.linspace(0.5, 1.8, len(all_days))
growth_probs = growth_weights / growth_weights.sum()
signup_dates = np.random.choice(all_days, size=n_users, p=growth_probs)

countries = np.random.choice(["US", "UK", "IT", "DE", "CA"], size=n_users, p=[0.4, 0.2, 0.15, 0.15, 0.1])
ad_channels = np.random.choice(["TikTok", "Google Ads", "Instagram", "Organic"], size=n_users, p=[0.3, 0.35, 0.2, 0.15])

users_df = pd.DataFrame({
    "user_id": user_ids,
    "signup_date": pd.to_datetime(signup_dates).date,
    "country": countries,
    "ad_channel": ad_channels
})

# 2. Simulate First 7 Days of App Activity (Events)
engagement_multiplier = users_df["ad_channel"].map({
    "Google Ads": 1.3,
    "Organic": 1.5,
    "Instagram": 1.0,
    "TikTok": 0.7
})

users_df["notes_created_7d"] = np.random.poisson(lam=4 * engagement_multiplier)
users_df["app_opens_7d"] = users_df["notes_created_7d"] + np.random.poisson(lam=3 * engagement_multiplier)
users_df["paywall_views_7d"] = np.random.binomial(n=3, p=np.clip(0.2 * engagement_multiplier, 0, 0.9))

# 3. Simulate Subscriptions & Revenue (6-Month LTV)
conversion_prob = np.clip(
    0.02 + (users_df["notes_created_7d"] * 0.015) + (users_df["paywall_views_7d"] * 0.04),
    0.01, 0.65
)
users_df["converted_to_paid"] = np.random.binomial(n=1, p=conversion_prob)

plan_types = np.where(np.random.rand(n_users) > 0.65, "Annual", "Monthly")
users_df["plan_type"] = np.where(users_df["converted_to_paid"] == 1, plan_types, "Free")

months_retained = np.random.randint(1, 7, size=n_users)
users_df["ltv_6m_gbp"] = np.where(
    users_df["plan_type"] == "Annual", 59.99,
    np.where(users_df["plan_type"] == "Monthly", months_retained * 9.99, 0.0)
)

# Overwrite the CSV file on your computer
users_df.to_csv("app_user_data.csv", index=False)

print("Updated dataset saved! Total users:", len(users_df))
users_df.head(5)

Updated dataset saved! Total users: 10000


,user_id,signup_date,country,ad_channel,notes_created_7d,app_opens_7d,paywall_views_7d,converted_to_paid,plan_type,ltv_6m_gbp
0,USER_00001,2026-04-04,US,Instagram,7,11,0,1,Annual,59.99
1,USER_00002,2026-06-25,US,TikTok,5,7,1,0,Free,0.00
2,USER_00003,2026-05-28,US,Google Ads,3,6,0,0,Free,0.00
3,USER_00004,2026-05-10,IT,Instagram,6,13,1,0,Free,0.00
4,USER_00005,2026-02-18,UK,Google Ads,4,11,0,0,Free,0.00


In [2]:
# Compare conversion rate and average 6-month LTV by ad channel
summary = users_df.groupby("ad_channel").agg(
    total_users=("user_id", "count"),
    conversion_rate=("converted_to_paid", "mean"),
    avg_ltv_gbp=("ltv_6m_gbp", "mean")
).round(3)

print(summary)

            total_users  conversion_rate  avg_ltv_gbp
ad_channel                                           
Google Ads         3553            0.139        6.008
Instagram          2031            0.114        5.452
Organic            1445            0.135        5.686
TikTok             2971            0.088        3.825


In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score

# 1. Load your dataset
df = pd.read_csv("app_user_data.csv")

# 2. Prepare features (X) and target (y)
# Convert text categories (country, ad_channel) into numbers using one-hot encoding
X = pd.get_dummies(df[["country", "ad_channel", "notes_created_7d", "app_opens_7d", "paywall_views_7d"]], drop_first=True)
y = df["converted_to_paid"]

# 3. Split into Training (80%) and Testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Train the model
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# 5. Evaluate model performance
predictions = model.predict(X_test)
probabilities = model.predict_proba(X_test)[:, 1]

print(f"Model Accuracy: {accuracy_score(y_test, predictions):.2%}")
print(f"ROC-AUC Score: {roc_auc_score(y_test, probabilities):.3f}")

# 6. Extract Feature Importance (What drives users to pay?)
feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Impact_Weight": model.coef_[0]
}).sort_values(by="Impact_Weight", ascending=False)

print("\nTop Drivers of Paid Conversion:")
print(feature_importance.head(6))

Model Accuracy: 87.50%
ROC-AUC Score: 0.637

Top Drivers of Paid Conversion:
             Feature  Impact_Weight
2   paywall_views_7d       0.438336
0   notes_created_7d       0.142901
5         country_UK       0.079542
4         country_IT       0.076422
9  ad_channel_TikTok       0.053030
6         country_US       0.052411


In [4]:
from scipy import stats

# 1. Simulate assigning users randomly to Control (A) or Treatment (B)
np.random.seed(42)
df["ab_group"] = np.random.choice(["Control_Old_Paywall", "Treatment_New_Paywall"], size=len(df))

# Give the Treatment group a realistic +2.5% lift in conversion rate
df["ab_converted"] = np.where(
    df["ab_group"] == "Treatment_New_Paywall",
    np.random.binomial(1, p=np.clip(df["converted_to_paid"].mean() + 0.025, 0, 1), size=len(df)),
    df["converted_to_paid"]
)

# 2. Summarise conversion rates by group
ab_summary = df.groupby("ab_group")["ab_converted"].agg(["count", "mean"]).rename(columns={"mean": "conversion_rate"})
print(ab_summary)

# 3. Run a Statistical Significance Test (Two-sample T-test)
control_data = df[df["ab_group"] == "Control_Old_Paywall"]["ab_converted"]
treatment_data = df[df["ab_group"] == "Treatment_New_Paywall"]["ab_converted"]

t_stat, p_value = stats.ttest_ind(treatment_data, control_data)
print(f"\nP-value: {p_value:.5f}")

if p_value < 0.05:
    print("DECISION: Statistically significant! Ship the new paywall to 100% of users.")
else:
    print("DECISION: Not statistically significant. Do not launch yet.")

                       count  conversion_rate
ab_group                                     
Control_Old_Paywall     5013         0.123479
Treatment_New_Paywall   4987         0.139964

P-value: 0.01479
DECISION: Statistically significant! Ship the new paywall to 100% of users.
